In [ ]:
import numpy as np
import pandas as pd
import io

# (SimplexEnginePandas クラスは既存のものを使用)

class SmartInputHandler:
    @staticmethod
    def get_list_input(prompt, expected_size):
        """スペース区切りで数値を一括入力させる"""
        while True:
            try:
                line = input(f"{prompt} (スペース区切りで{expected_size}個入力): ")
                values = [float(x) for x in line.split()]
                if len(values) != expected_size:
                    print(f"エラー: {expected_size}個の数値を入力してください。")
                    continue
                return values
            except ValueError:
                print("エラー: 有効な数値を入力してください。")

    @staticmethod
    def get_csv_input():
        """CSV形式のテキストを貼り付けて入力させる"""
        print("\n[CSVモード] Excel等のデータを以下に貼り付け、Enterを2回押してください:")
        print("格式: x1, x2, ..., xn, type, b")
        lines = []
        while True:
            line = input()
            if not line: break
            lines.append(line)

        csv_data = "\n".join(lines)
        df = pd.read_csv(io.StringIO(csv_data), header=None)
        return df

def interactive_input_v2():
    print("="*40)
    print("   線形計画法 統合入力システム (V2)")
    print("="*40)

    mode = input("入力モードを選んでください (1:対話, 2:一括(スペース区切り), 3:CSV貼り付け): ")

    if mode == '1':
        # 既存の対話型ロジック
        n = int(input("変数の数 (n): "))
        m = int(input("制約の数 (m): "))
        c = [float(input(f"c{i+1}: ")) for i in range(n)]
        A, b, types = [], [], []
        for i in range(m):
            print(f"--- 制約 {i+1} ---")
            A.append([float(input(f"  x{j+1}係数: ")) for j in range(n)])
            types.append(input("  タイプ (<=, >=, =): "))
            b.append(float(input("  右辺 b: ")))

    elif mode == '2':
        n = int(input("変数の数 (n): "))
        m = int(input("制約の数 (m): "))
        print("\n目的関数 c を入力してください")
        c = SmartInputHandler.get_list_input("目的係数", n)
        A, b, types = [], [], []
        for i in range(m):
            print(f"\n--- 制約 {i+1} ---")
            A.append(SmartInputHandler.get_list_input(f"係数行", n))
            types.append(input("  タイプ (<=, >=, =): "))
            b.append(float(input("  右辺 b: ")))

    elif mode == '3':
        df = SmartInputHandler.get_csv_input()
        # 最後の2列が type と b と仮定
        c_line = input("目的関数 c をスペース区切りで入力: ")
        c = [float(x) for x in c_line.split()]
        n = len(c)
        A = df.iloc[:, :n].values.tolist()
        types = df.iloc[:, n].tolist()
        b = df.iloc[:, n+1].tolist()
        m = len(A)

    # 計算実行
    solver = SimplexEnginePandas(c, A, b, types)
    result = solver.solve()

    # 結果レポート表示 (既存の表示ロジック)
    if isinstance(result, tuple):
        df_v, df_c, obj = result
        print("\n" + "★"*20)
        print(f" 最適化結果: z = {obj:.4f}")
        print("★"*20)
        print("\n[最適解]\n", df_v.to_string(index=False))
        print("\n[感度分析]\n", df_c.to_string(index=False))
    else:
        print("\nエラー:", result)

if __name__ == "__main__":
    interactive_input_v2()


class SimplexEnginePandas:
    def __init__(self, c, A, b, constraint_types):
        self.c = np.array(c, dtype=float)
        self.A = np.array(A, dtype=float)
        self.b = np.array(b, dtype=float)
        self.types = constraint_types
        self.m, self.n = self.A.shape
        self.basis = [0] * self.m

    def solve(self):
        # 1. パラメータ準備
        num_slack = sum(1 for t in self.types if t in ['<=', '>='])
        num_artif = sum(1 for t in self.types if t in ['>=', '='])
        total_vars = self.n + num_slack + num_artif

        self.tableau = np.zeros((self.m + 2, total_vars + 1))
        slack_idx, artif_idx = self.n, self.n + num_slack

        # 2. タブロー構築
        for i in range(self.m):
            row = i + 2
            self.tableau[row, :self.n] = self.A[i]
            self.tableau[row, -1] = self.b[i]
            if self.types[i] == '<=':
                self.tableau[row, slack_idx] = 1
                self.basis[i] = slack_idx
                slack_idx += 1
            elif self.types[i] == '>=':
                self.tableau[row, slack_idx] = -1
                self.tableau[row, artif_idx] = 1
                self.basis[i] = artif_idx
                slack_idx += 1; artif_idx += 1
            elif self.types[i] == '=':
                self.tableau[row, artif_idx] = 1
                self.basis[i] = artif_idx
                artif_idx += 1

        # 3. Phase 1 & 2 実行
        self.tableau[0, :self.n] = self.c
        if num_artif > 0:
            for i in range(self.m):
                if self.types[i] in ['>=', '=']: self.tableau[1] -= self.tableau[i+2]
            if self._optimize(1) == "非有界" or self.tableau[1, -1] < -1e-9: return "実行不能"

        self.tableau = np.delete(self.tableau, range(self.n + num_slack, total_vars), axis=1)
        self.tableau = np.delete(self.tableau, 1, axis=0)

        status = self._optimize(0)
        return self._format_results(status, num_slack) if status == "最適" else status

    def _optimize(self, target_row):
        while True:
            pivot_col = -1
            for j in range(self.tableau.shape[1] - 1):
                if self.tableau[target_row, j] < -1e-9:
                    pivot_col = j; break
            if pivot_col == -1: return "最適"

            ratios = [self.tableau[i+1, -1] / self.tableau[i+1, pivot_col]
                      if self.tableau[i+1, pivot_col] > 1e-9 else np.inf for i in range(self.m)]
            pivot_row_idx = np.argmin(ratios)
            if ratios[pivot_row_idx] == np.inf: return "非有界"

            actual_row = pivot_row_idx + 1
            self.tableau[actual_row, :] /= self.tableau[actual_row, pivot_col]
            for i in range(self.tableau.shape[0]):
                if i != actual_row: self.tableau[i, :] -= self.tableau[i, pivot_col] * self.tableau[actual_row, :]
            self.basis[pivot_row_idx] = pivot_col

    def _format_results(self, status, num_slack):
        # 変数結果のDF
        x_sol = np.zeros(self.n)
        for i, b_idx in enumerate(self.basis):
            if b_idx < self.n: x_sol[b_idx] = self.tableau[i+1, -1]

        df_vars = pd.DataFrame({'変数': [f'x{i+1}' for i in range(self.n)], '最適値': x_sol})

        # 制約結果のDF (潜在価格とスラック)
        shadow_prices = self.tableau[0, self.n : self.n + num_slack]
        # スラック値の抽出
        slack_values = np.zeros(num_slack)
        for i, b_idx in enumerate(self.basis):
            if self.n <= b_idx < self.n + num_slack:
                slack_values[b_idx - self.n] = self.tableau[i+1, -1]

        df_cons = pd.DataFrame({
            '制約No': [f'Constraint {i+1}' for i in range(self.m)],
            'タイプ': self.types,
            '潜在価格 (Shadow Price)': shadow_prices,
            '余裕値 (Slack/Surplus)': slack_values
        })

        return df_vars, df_cons, self.tableau[0, -1]

# --- ランダム生成・実行・表示 ---
def run_enhanced_simulation():
    n, m = 3, 3 # 固定またはランダム
    c = np.random.randint(1, 10, n) # 最小化問題なので正の値
    A = np.random.randint(1, 10, (m, n))
    b = np.random.randint(20, 100, m)
    types = ['>='] * m # 栄養問題のような形式

    solver = SimplexEnginePandas(c, A, b, types)
    result = solver.solve()

    if isinstance(result, tuple):
        df_v, df_c, obj = result
        print("\n=== 最適化報告書 ===")
        print(f"最小化された目的関数値: {obj:.4f}")
        print("\n[決定変数の状態]")
        print(df_v.to_string(index=False))
        print("\n[制約条件の状態と感度分析]")
        print(df_c.to_string(index=False))
    else:
        print(f"結果: {result}")

run_enhanced_simulation()

   線形計画法 統合入力システム (V2)
入力モードを選んでください (1:対話, 2:一括(スペース区切り), 3:CSV貼り付け): 2
変数の数 (n): 4
制約の数 (m): 5

目的関数 c を入力してください
目的係数 (スペース区切りで4個入力): 2.4 8 3
エラー: 4個の数値を入力してください。
目的係数 (スペース区切りで4個入力): 2 6 7 8

--- 制約 1 ---
係数行 (スペース区切りで4個入力): 1 2 3 4
  タイプ (<=, >=, =): <=
  右辺 b: 9 7 5 4


ValueError: could not convert string to float: '9 7 5 4'

In [ ]:
import numpy as np

class UniversalSimplexSolver:
    def __init__(self, c, A, b, constraint_types):
        self.c = np.array(c, dtype=float)
        self.A = np.array(A, dtype=float)
        self.b = np.array(b, dtype=float)
        self.types = constraint_types  # list of '<=', '>=', '='
        self.m, self.n = self.A.shape
        self.tableau = None

    def solve(self):
        # 1. 右辺bが負の場合、正にする（制約を反転）
        for i in range(self.m):
            if self.b[i] < 0:
                self.A[i] *= -1
                self.b[i] *= -1
                if self.types[i] == '<=': self.types[i] = '>='
                elif self.types[i] == '>=': self.types[i] = '<='

        # 2. 標準形と人工変数の追加（2段階法準備）
        # 変数構成: 元の変数(n) + スラック(s) + 人工変数(a)
        num_slack = self.types.count('<=') + self.types.count('>=')
        num_artif = self.types.count('>=') + self.types.count('=')

        total_vars = self.n + num_slack + num_artif
        self.tableau = np.zeros((self.m + 2, total_vars + 1)) # +2は目的関数行と人工目的関数行

        slack_idx = self.n
        artif_idx = self.n + num_slack
        artif_rows = []

        for i in range(self.m):
            self.tableau[i+2, :self.n] = self.A[i]
            if self.types[i] == '<=':
                self.tableau[i+2, slack_idx] = 1
                slack_idx += 1
            elif self.types[i] == '>=':
                self.tableau[i+2, slack_idx] = -1
                self.tableau[i+2, artif_idx] = 1
                artif_rows.append(i+2)
                slack_idx += 1
                artif_idx += 1
            elif self.types[i] == '=':
                self.tableau[i+2, artif_idx] = 1
                artif_rows.append(i+2)
                artif_idx += 1
            self.tableau[i+2, -1] = self.b[i]

        # 第1段階の目的関数 (人工変数の和を最小化)
        for r in artif_rows:
            self.tableau[1] -= self.tableau[r]
        self.tableau[1, self.n + num_slack : total_vars] = 0 # 人工変数列を0に整地

        # 第2段階の目的関数 (元の最小化問題: 係数そのまま)
        self.tableau[0, :self.n] = self.c

        # --- Phase 1: 人工変数を排除 ---
        if num_artif > 0:
            self._optimize(phase=1)
            if abs(self.tableau[1, -1]) > 1e-9:
                return "実行可能解なし"

        # --- Phase 2: 最適化 ---
        # 人工変数列を削除
        self.tableau = np.delete(self.tableau, range(self.n + num_slack, total_vars), axis=1)
        self.tableau = np.delete(self.tableau, 1, axis=0) # 人工目的関数行を削除
        return self._optimize(phase=2)

    def _optimize(self, phase):
        while True:
            # Blandのルール: 負の係数のうち最小の添字を選択
            pivot_col = -1
            for j in range(self.tableau.shape[1] - 1):
                if self.tableau[0, j] < -1e-9: # 最小化問題での判定
                    pivot_col = j
                    break

            if pivot_col == -1: break # 最適

            # 比率テスト
            ratios = []
            for i in range(1, self.tableau.shape[0]):
                val = self.tableau[i, pivot_col]
                if val > 1e-9:
                    ratios.append(self.tableau[i, -1] / val)
                else:
                    ratios.append(np.inf)

            pivot_row = np.argmin(ratios) + 1
            if ratios[pivot_row-1] == np.inf:
                return "非有界"

            # ピボット操作
            self.tableau[pivot_row, :] /= self.tableau[pivot_row, pivot_col]
            for i in range(self.tableau.shape[0]):
                if i != pivot_row:
                    self.tableau[i, :] -= self.tableau[i, pivot_col] * self.tableau[pivot_row, :]

        return self.tableau[0, -1], self.tableau[1:, -1]

def generate_random_problem():
    """問題をランダムに生成する"""
    n = np.random.randint(2, 6)  # 変数 2~5
    m = np.random.randint(2, 6)  # 制約 2~5

    c = np.random.uniform(-10, 10, size=n)
    A = np.random.uniform(-5, 15, size=(m, n))
    b = np.random.uniform(5, 50, size=m)
    types = np.random.choice(['<=', '>=', '='], size=m)

    print(f"--- ランダム生成問題 (変数:{n}, 制約:{m}) ---")
    print(f"目的関数係数 c: {c.round(2)}")
    print(f"制約タイプ: {types}")
    return c, A, b, types

# 実行
c, A, b, types = generate_random_problem()
solver = UniversalSimplexSolver(c, A, b, types)
result = solver.solve()

if isinstance(result, tuple):
    val, sol = result
    print(f"最適値: {val:.4f}")
else:
    print(f"結果: {result}")

In [ ]:
import numpy as np

class SimplexEngine:
    def __init__(self, c, A, b, constraint_types):
        self.c = np.array(c, dtype=float)
        self.A = np.array(A, dtype=float)
        self.b = np.array(b, dtype=float)
        # リストのまま保持するか、NumPyの比較機能を使う
        self.types = constraint_types
        self.m, self.n = self.A.shape
        self.tableau = None
        self.basis = [] # 現在の基底変数のインデックス

    def solve(self):
        # --- 修正箇所: countメソッドの代替 ---
        num_slack = sum(1 for t in self.types if t in ['<=', '>='])
        num_artif = sum(1 for t in self.types if t in ['>=', '='])

        total_vars = self.n + num_slack + num_artif
        # 行: 目的関数, 人工目的関数, 各制約
        self.tableau = np.zeros((self.m + 2, total_vars + 1))

        # 変数インデックスの管理
        slack_idx = self.n
        artif_idx = self.n + num_slack
        self.basis = []

        # タブロー構築
        for i in range(self.m):
            row_idx = i + 2
            self.tableau[row_idx, :self.n] = self.A[i]

            if self.types[i] == '<=':
                self.tableau[row_idx, slack_idx] = 1
                self.basis.append(slack_idx)
                slack_idx += 1
            elif self.types[i] == '>=':
                self.tableau[row_idx, slack_idx] = -1
                self.tableau[row_idx, artif_idx] = 1
                self.basis.append(artif_idx)
                slack_idx += 1
                artif_idx += 1
            elif self.types[i] == '=':
                self.tableau[row_idx, artif_idx] = 1
                self.basis.append(artif_idx)
                artif_idx += 1
            self.tableau[row_idx, -1] = self.b[i]

        # 第1段階: 人工変数の和の最小化
        for i in range(self.m):
            if self.types[i] in ['>=', '=']:
                self.tableau[1] -= self.tableau[i+2]

        # 第2段階用目的関数
        self.tableau[0, :self.n] = self.c

        # --- Phase 1 ---
        if num_artif > 0:
            res = self._optimize(target_row=1)
            if self.tableau[1, -1] < -1e-9:
                return "実行可能解なし"

        # --- Phase 2 ---
        # 人工変数列を削除
        self.tableau = np.delete(self.tableau, range(self.n + num_slack, total_vars), axis=1)
        self.tableau = np.delete(self.tableau, 1, axis=0)
        return self._optimize(target_row=0)

    def _optimize(self, target_row):
        while True:
            # Blandのルール (最小添字)
            pivot_col = -1
            for j in range(self.tableau.shape[1] - 1):
                if self.tableau[target_row, j] < -1e-9:
                    pivot_col = j
                    break

            if pivot_col == -1: break

            # 比率テスト
            ratios = []
            for i in range(1, self.tableau.shape[0]):
                val = self.tableau[i, pivot_col]
                ratios.append(self.tableau[i, -1] / val if val > 1e-9 else np.inf)

            pivot_row = np.argmin(ratios) + 1
            if ratios[pivot_row-1] == np.inf: return "非有界"

            # ピボット操作
            self.tableau[pivot_row, :] /= self.tableau[pivot_row, pivot_col]
            for i in range(self.tableau.shape[0]):
                if i != pivot_row:
                    self.tableau[i, :] -= self.tableau[i, pivot_col] * self.tableau[pivot_row, :]

            # 基底の更新
            self.basis[pivot_row-1] = pivot_col

        return self.tableau[0, -1], self.get_solution()

    def get_solution(self):
        sol = np.zeros(self.n)
        for i, b_idx in enumerate(self.basis):
            if b_idx < self.n:
                sol[b_idx] = self.tableau[i+1, -1]
        return sol

# --- ランダム生成と実行 ---
def run_random_simulation():
    n_vars = np.random.randint(2, 4)
    n_cons = np.random.randint(2, 4)
    c = np.random.randint(-10, 10, n_vars)
    A = np.random.randint(-5, 15, (n_cons, n_vars))
    b = np.random.randint(10, 50, n_cons)
    types = np.random.choice(['<=', '>=', '='], n_cons)

    print(f"Random LP Problem: Minimize {c}x subject to Ax {types} b")

    engine = SimplexEngine(c, A, b, types)
    result = engine.solve()

    if isinstance(result, tuple):
        val, x = result
        print(f"Status: Optimal")
        print(f"Objective Value: {val:.2f}")
        print(f"Solution: {x}")
    else:
        print(f"Status: {result}")

run_random_simulation()

In [ ]:
import numpy as np

class SimplexEngine:
    def __init__(self, c, A, b, constraint_types):
        self.c = np.array(c, dtype=float)
        self.A = np.array(A, dtype=float)
        self.b = np.array(b, dtype=float)
        self.types = constraint_types
        self.m, self.n = self.A.shape
        self.tableau = None
        # 各行(1~m)に対応する基底変数のインデックスを保持
        self.basis = [0] * self.m

    def solve(self):
        # 1. 準備：スラック変数と人工変数の数をカウント
        num_slack = sum(1 for t in self.types if t in ['<=', '>='])
        num_artif = sum(1 for t in self.types if t in ['>=', '='])
        total_vars = self.n + num_slack + num_artif

        # タブロー: [0]目的関数, [1]人工目的関数, [2~m+1]制約
        self.tableau = np.zeros((self.m + 2, total_vars + 1))

        slack_idx = self.n
        artif_idx = self.n + num_slack

        # 2. タブローの構築
        for i in range(self.m):
            row = i + 2
            self.tableau[row, :self.n] = self.A[i]
            self.tableau[row, -1] = self.b[i]

            if self.types[i] == '<=':
                self.tableau[row, slack_idx] = 1
                self.basis[i] = slack_idx
                slack_idx += 1
            elif self.types[i] == '>=':
                self.tableau[row, slack_idx] = -1
                self.tableau[row, artif_idx] = 1
                self.basis[i] = artif_idx
                slack_idx += 1
                artif_idx += 1
            elif self.types[i] == '=':
                self.tableau[row, artif_idx] = 1
                self.basis[i] = artif_idx
                artif_idx += 1

        # 3. Phase 1: 人工目的関数の設定 (人工変数の行を足し合わせる)
        for i in range(self.m):
            if self.types[i] in ['>=', '=']:
                self.tableau[1] -= self.tableau[i + 2]

        self.tableau[0, :self.n] = self.c

        # --- Phase 1 実行 ---
        if num_artif > 0:
            status = self._optimize(target_row=1)
            if self.tableau[1, -1] < -1e-9:
                return "実行可能解なし"

        # --- Phase 2 実行 ---
        # 人工変数列を削除し、第1段階用行を削除
        self.tableau = np.delete(self.tableau, range(self.n + num_slack, total_vars), axis=1)
        self.tableau = np.delete(self.tableau, 1, axis=0)

        final_status = self._optimize(target_row=0)
        if final_status == "非有界": return final_status

        return self._generate_report(num_slack)

    def _optimize(self, target_row):
        while True:
            # Blandのルール
            pivot_col = -1
            for j in range(self.tableau.shape[1] - 1):
                if self.tableau[target_row, j] < -1e-9:
                    pivot_col = j
                    break

            if pivot_col == -1: break

            # 比率テスト
            ratios = []
            for i in range(1, self.m + 1):
                val = self.tableau[i + (1 if target_row == 0 else 0), pivot_col]
                ratios.append(self.tableau[i + (1 if target_row == 0 else 0), -1] / val if val > 1e-9 else np.inf)

            pivot_row_idx = np.argmin(ratios)
            if ratios[pivot_row_idx] == np.inf: return "非有界"

            # ピボット操作
            actual_row = pivot_row_idx + (1 if target_row == 0 else 0)
            self.tableau[actual_row, :] /= self.tableau[actual_row, pivot_col]
            for i in range(self.tableau.shape[0]):
                if i != actual_row:
                    self.tableau[i, :] -= self.tableau[i, pivot_col] * self.tableau[actual_row, :]

            # 基底変数のインデックスを更新
            self.basis[pivot_row_idx] = pivot_col

        return "最適"

    def _generate_report(self, num_slack):
        # 解の抽出
        x_sol = np.zeros(self.n)
        for i, b_idx in enumerate(self.basis):
            if b_idx < self.n:
                x_sol[b_idx] = self.tableau[i+1, -1]

        # 潜在価格 (Shadow Prices) の抽出: スラック変数列の指示数
        shadow_prices = self.tableau[0, self.n : self.n + num_slack]

        return {
            "obj_val": self.tableau[0, -1],
            "solution": x_sol,
            "shadow_prices": shadow_prices
        }

# --- ランダムシミュレーション ---
def run_simulation():
    # 規模をランダム決定 (変数3~5, 制約2~4)
    n, m = np.random.randint(3, 6), np.random.randint(2, 5)
    c = np.random.uniform(-10, 10, n)
    A = np.random.uniform(-5, 10, (m, n))
    b = np.random.uniform(10, 50, m)
    types = np.random.choice(['<=', '>=', '='], m)

    print(f"--- Randomized Problem (n={n}, m={m}) ---")
    solver = SimplexEngine(c, A, b, types)
    res = solver.solve()

    if isinstance(res, dict):
        print(f"Optimal Value: {res['obj_val']:.4f}")
        print(f"Solution x: {res['solution'].round(4)}")
        print(f"Shadow Prices: {res['shadow_prices'].round(4)}")
    else:
        print(f"Result: {res}")

run_simulation()